# Estática aplicada a obras civiles · Cuaderno de cálculo

**Curso de Estática · Tecnología en Obras Civiles · Universidad del Pacífico, sede Guapi**
Autor: Jerson Enrique Sinisterra García · Prototipo 1.0 · 2026

Este cuaderno es la versión en Python del prototipo web del curso
([ver el prototipo](https://jerson0915.github.io/estatica-obra/curso/)). Tiene las mismas 12 unidades.
Cada una trae:

1. **Explicación** del tema y de las fórmulas que se usan.
2. **Código comentado**, para que se entienda qué hace cada línea.
3. **Un ejemplo resuelto** con números, que se puede cambiar.
4. **Un control interactivo** (deslizadores) en las unidades donde ayuda a ver cómo cambia el resultado.

**Cómo usarlo en Colab:** ejecuta las celdas en orden, de arriba abajo, con `Shift + Enter`.
La primera celda de código carga las librerías y las funciones de dibujo que usan todas las unidades.
Si cambias un número en un ejemplo, vuelve a ejecutar esa celda.

**Convenciones:** unidades del Sistema Internacional (kN, m, kN·m); derecha y arriba positivos;
momento antihorario positivo; en armaduras, tensión positiva y compresión negativa.

## 0 · Configuración

Esta celda carga las tres librerías que usa todo el cuaderno:

- **NumPy** (`np`): cálculo con vectores y matrices.
- **Matplotlib** (`plt`): gráficas y dibujos.
- **ipywidgets** (`interact`): deslizadores para cambiar datos y ver el resultado al instante. Colab ya la trae instalada.

También define funciones pequeñas que se repiten en varias unidades: convertir grados a radianes y dibujar flechas.

In [ ]:
# --- Librerías ---
import numpy as np                      # vectores, matrices, trigonometría
import matplotlib.pyplot as plt         # gráficas
from ipywidgets import interact, FloatSlider, IntSlider, Dropdown   # controles interactivos

# --- Estilo de las gráficas: cuadrícula suave y letra legible ---
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3, "font.size": 10})

# Colores del curso (los mismos de la página web), uno por cada parte
AZUL, VERDE, NARANJA, MORADO = "#2463d6", "#0b8f7e", "#e2580b", "#8b3fd1"
ROJO_TENSION, AZUL_COMPRESION = "#c0262d", "#1d5fd1"

def rad(grados):
    # Las funciones trigonométricas de NumPy trabajan en radianes: este es el equivalente
    # a poner la calculadora en modo DEG.
    return np.deg2rad(grados)

def flecha(ax, x0, y0, x1, y1, color, texto=None, ancho=0.012):
    # Dibuja una flecha de (x0, y0) a (x1, y1) y, si se pide, una etiqueta en la punta.
    ax.annotate("", xy=(x1, y1), xytext=(x0, y0),
                arrowprops=dict(arrowstyle="-|>", color=color, lw=2.2, mutation_scale=14))
    if texto:
        ax.text(x1, y1, " " + texto, color=color, fontsize=10, fontweight="bold", va="bottom")

print("Listo: librerías y funciones cargadas.")

# Parte I · Fundamentos

## 01 · ¿Qué es la estática?

La estática estudia los cuerpos en **equilibrio**: en reposo o moviéndose a velocidad constante.
La condición es que la suma de fuerzas sea cero. Si no lo es, el cuerpo se acelera (eso es dinámica).

$$\sum \vec F = 0 \iff \vec a = 0$$

**Qué hace el código:** dos obreros jalan un bloque en sentidos opuestos. La función suma las dos fuerzas
(la de la izquierda con signo negativo) y dice si el bloque queda en equilibrio o se acelera.

In [ ]:
def tira_y_afloja(F_izquierda, F_derecha):
    # La fuerza hacia la izquierda es negativa y la de la derecha positiva (convención de signos).
    F_neta = F_derecha - F_izquierda
    if F_neta == 0:
        return f"ΣF = 0 N → el bloque está en equilibrio: ESTÁTICA"
    sentido = "derecha" if F_neta > 0 else "izquierda"
    return f"ΣF = {abs(F_neta)} N hacia la {sentido} → el bloque se acelera: DINÁMICA"

print(tira_y_afloja(60, 60))   # fuerzas iguales
print(tira_y_afloja(60, 75))   # gana el obrero de la derecha

## 02 · Unidades y conversiones

La NSR-10 trabaja en el Sistema Internacional (m, kN, MPa), pero en la obra se habla de pulgadas, libras y psi.

- **Convertir es multiplicar por un factor.** Cada unidad se guarda con su valor en la unidad base (m, N, kg, Pa).
  Para pasar de A a B se multiplica por `factor[A] / factor[B]`.
- **Masa no es peso.** El peso es una fuerza: $W = m \cdot g$, con $g = 9{,}81\ \text{m/s}^2$.
  1 kN es el peso de unos 102 kg, más o menos dos bultos de cemento de 50 kg.

In [ ]:
# Cada diccionario guarda cuánto vale cada unidad en la unidad base de su magnitud.
FACTORES = {
    "longitud": {"m": 1, "cm": 0.01, "mm": 0.001, "ft": 0.3048, "in": 0.0254},
    "fuerza":   {"N": 1, "kN": 1000, "kgf": 9.80665, "lbf": 4.4482216, "kip": 4448.2216},
    "masa":     {"kg": 1, "t": 1000, "lb": 0.45359237},
    "presion":  {"Pa": 1, "kPa": 1e3, "MPa": 1e6, "psi": 6894.757, "ksi": 6894757, "kgf/cm2": 98066.5},
    "momento":  {"N·m": 1, "kN·m": 1000, "kgf·m": 9.80665, "lbf·ft": 1.3558179, "kip·ft": 1355.8179},
}

def convertir(valor, magnitud, de, a):
    # factor = cuántas unidades "a" caben en una unidad "de"
    tabla = FACTORES[magnitud]
    factor = tabla[de] / tabla[a]
    return valor * factor, factor

# Casos de obra
casos = [(0.5, "longitud", "in", "mm", 'Varilla de 1/2"'),
         (3000, "presion", "psi", "MPa", "Concreto de 3 000 psi"),
         (60, "presion", "ksi", "MPa", "Acero fy = 60 ksi"),
         (1000, "fuerza", "kgf", "kN", "1 000 kgf"),
         (50, "masa", "kg", "lb", "Bulto de cemento de 50 kg")]
for valor, mag, de, a, nombre in casos:
    r, f = convertir(valor, mag, de, a)
    print(f"{nombre:28s}: {valor} {de} × {f:.6g} = {r:.4g} {a}")

# Masa y peso
m = 50                       # kg
W = m * 9.81 / 1000          # kN
print(f"\nUn bulto de {m} kg pesa W = m·g = {W:.3f} kN")

## 03 · Vectores fuerza en el plano

Una fuerza tiene magnitud y dirección. Para sumarlas se **descompone** cada una en x y y,
se suman las componentes por separado y se **recompone** la resultante:

$$F_x = F\cos\theta \qquad F_y = F\sin\theta \qquad R = \sqrt{(\textstyle\sum F_x)^2 + (\sum F_y)^2} \qquad \theta_R = \operatorname{atan2}(R_y, R_x)$$

La **equilibrante** $E = -R$ es la fuerza que deja el sistema en equilibrio.

**Qué hace el código:** recibe una lista de fuerzas (magnitud y ángulo medido desde +x), calcula la resultante
y dibuja el polígono de fuerzas (cada flecha empieza donde termina la anterior). Se usa `atan2` y no `atan`
porque `atan2` mira el signo de las dos componentes y da el ángulo en el cuadrante correcto.

In [ ]:
def resultante(fuerzas):
    # fuerzas: lista de (magnitud, ángulo en grados)
    F = np.array([m for m, a in fuerzas], dtype=float)
    th = rad(np.array([a for m, a in fuerzas], dtype=float))
    Fx, Fy = F * np.cos(th), F * np.sin(th)        # componentes de cada fuerza
    Rx, Ry = Fx.sum(), Fy.sum()                     # suma de componentes
    R = np.hypot(Rx, Ry)                            # magnitud de la resultante
    thR = np.degrees(np.arctan2(Ry, Rx)) % 360      # ángulo en el cuadrante correcto
    return Fx, Fy, Rx, Ry, R, thR

def dibujar_resultante(fuerzas, nombres=None, equilibrante=True):
    Fx, Fy, Rx, Ry, R, thR = resultante(fuerzas)
    fig, ax = plt.subplots(figsize=(5.5, 5.5))
    colores = [AZUL, VERDE, MORADO, NARANJA]
    x = y = 0
    for i, (fx, fy) in enumerate(zip(Fx, Fy)):        # polígono de fuerzas: punta con cola
        flecha(ax, x, y, x + fx, y + fy, colores[i % 4], (nombres or [f"F{i+1}" for i in range(len(Fx))])[i])
        x, y = x + fx, y + fy
    flecha(ax, 0, 0, Rx, Ry, "black", "R")
    if equilibrante:
        flecha(ax, 0, 0, -Rx, -Ry, "gray", "E = −R")
    lim = 1.15 * max(np.abs(np.r_[np.cumsum(Fx), np.cumsum(Fy), Rx, Ry]).max(), 1)
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim); ax.set_aspect("equal")
    ax.axhline(0, color="k", lw=0.8); ax.axvline(0, color="k", lw=0.8)
    ax.set_title(f"R = {R:.2f} N a {thR:.1f}°"); plt.show()
    for i, (fx, fy) in enumerate(zip(Fx, Fy)):
        print(f"F{i+1}: Fx = {fx:8.2f}  Fy = {fy:8.2f}")
    print(f"R : Rx = {Rx:8.2f}  Ry = {Ry:8.2f}   |R| = {R:.2f} N  θ = {thR:.1f}°")

# Ejemplo: tres tensores sobre un poste (vista en planta)
dibujar_resultante([(70, 20), (60, 150), (55, 255)], ["A", "B", "C"])

**Pruébalo:** mueve los deslizadores. Cuando la resultante sea casi cero, el poste está en equilibrio.

In [ ]:
@interact(F1=FloatSlider(value=60, min=0, max=100, step=1, description="F1 (N)"),
          a1=IntSlider(value=30, min=0, max=360, description="θ1 (°)"),
          F2=FloatSlider(value=45, min=0, max=100, step=1, description="F2 (N)"),
          a2=IntSlider(value=120, min=0, max=360, description="θ2 (°)"))
def _(F1, a1, F2, a2):
    dibujar_resultante([(F1, a1), (F2, a2)], equilibrante=False)

## 04 · Vectores fuerza en el espacio

Un cable tensor sale de un punto A del poste y llega a un anclaje B en el piso. Su fuerza se escribe con el
**vector unitario** de la recta AB:

$$\vec r_{AB} = B - A \qquad \vec u = \frac{\vec r_{AB}}{|\vec r_{AB}|} \qquad \vec F = F\,\vec u$$

Los **ángulos directores** cumplen $\cos\alpha = u_x$, $\cos\beta = u_y$, $\cos\gamma = u_z$ y
$\cos^2\alpha + \cos^2\beta + \cos^2\gamma = 1$ (sirve para comprobar).

**Qué hace el código:** calcula el vector posición, el unitario, las tres componentes de la fuerza, los ángulos
directores, y cuánto comprime el cable al poste (|Fz|) y cuánto lo empuja hacia el anclaje (componente horizontal).
Al final dibuja la escena en 3D.

In [ ]:
def fuerza_cable(A, B, F):
    A, B = np.array(A, float), np.array(B, float)
    r = B - A                          # vector posición: va desde donde actúa (A) hacia donde jala (B)
    L = np.linalg.norm(r)              # longitud del cable
    u = r / L                          # vector unitario
    Fv = F * u                         # fuerza en componentes (kN)
    ang = np.degrees(np.arccos(u))     # ángulos directores α, β, γ
    print(f"r_AB = {np.round(r, 3)} m   |r_AB| = {L:.3f} m")
    print(f"u    = {np.round(u, 4)}")
    print(f"F    = ({Fv[0]:.3f}; {Fv[1]:.3f}; {Fv[2]:.3f}) kN")
    print(f"α = {ang[0]:.1f}°  β = {ang[1]:.1f}°  γ = {ang[2]:.1f}°   comprobación Σcos² = {np.sum(u**2):.4f}")
    print(f"Compresión que agrega al poste |Fz| = {abs(Fv[2]):.3f} kN")
    print(f"Empuje horizontal hacia el anclaje = {np.hypot(Fv[0], Fv[1]):.3f} kN")
    # Dibujo 3D
    fig = plt.figure(figsize=(6, 5)); ax = fig.add_subplot(projection="3d")
    ax.plot([0, 0], [0, 0], [0, A[2]], color="k", lw=5, label="poste")
    ax.plot([A[0], B[0]], [A[1], B[1]], [A[2], B[2]], color="gray", lw=2, label="cable")
    esc = min(L * 0.5, F * 0.45)
    ax.quiver(*A, *(u * esc), color=NARANJA, lw=3, label=f"F = {F} kN")
    ax.scatter(*B, color="k"); ax.text(*B, " B"); ax.text(*A, " A")
    ax.set_xlabel("x (m)"); ax.set_ylabel("y (m)"); ax.set_zlabel("z (m)"); ax.legend(); plt.show()
    return Fv

# Ejemplo: amarre a 12 m de altura, anclaje en (6; 4; 0) m, tensión de 10 kN
fuerza_cable(A=(0, 0, 12), B=(6, 4, 0), F=10);

# Parte II · Equilibrio

## 05 · Equilibrio de una partícula

Un andamio de peso W cuelga de un anillo sostenido por dos cables con ángulos α y β desde la horizontal.
El **diagrama de cuerpo libre (DCL)** del anillo tiene tres fuerzas: T₁, T₂ y W. Las dos ecuaciones de equilibrio son:

$$\sum F_x = -T_1\cos\alpha + T_2\cos\beta = 0 \qquad \sum F_y = T_1\sin\alpha + T_2\sin\beta - W = 0$$

**Qué hace el código:** escribe esas dos ecuaciones como un sistema matricial $[A]\{T\} = \{b\}$ y lo resuelve con
`np.linalg.solve`. Así se ve que "resolver equilibrio" es resolver un sistema de ecuaciones lineales. Después lo compara
con la fórmula cerrada $T_1 = W\cos\beta/\sin(\alpha+\beta)$.

In [ ]:
def nudo_dos_cables(W, alfa, beta):
    a, b = rad(alfa), rad(beta)
    # Fila 1: ΣFx = 0 → −cos α·T1 + cos β·T2 = 0
    # Fila 2: ΣFy = 0 →  sen α·T1 + sen β·T2 = W
    A = np.array([[-np.cos(a), np.cos(b)],
                  [ np.sin(a), np.sin(b)]])
    rhs = np.array([0, W])
    T1, T2 = np.linalg.solve(A, rhs)
    # Fórmula cerrada, para comprobar
    T1f = W * np.cos(b) / np.sin(a + b)
    T2f = W * np.cos(a) / np.sin(a + b)
    print(f"W = {W} kN, α = {alfa}°, β = {beta}°")
    print(f"T1 = {T1:.3f} kN (fórmula: {T1f:.3f})   T2 = {T2:.3f} kN (fórmula: {T2f:.3f})")
    print(f"Cada cable lleva {T1/W*100:.0f} % y {T2/W*100:.0f} % del peso")
    return T1, T2

nudo_dos_cables(W=12, alfa=45, beta=45)
nudo_dos_cables(W=12, alfa=30, beta=30)   # cables más acostados → más tensión
nudo_dos_cables(W=12, alfa=60, beta=60);

**Gráfica:** cómo crece la tensión al acostar los cables (α = β). A 30° cada cable carga el peso completo.

In [ ]:
ang = np.linspace(10, 85, 200)
T = 1 / (2 * np.sin(rad(ang)))        # tensión de cada cable dividida por W (caso simétrico)
plt.figure(figsize=(6, 3.5))
plt.plot(ang, T, color=VERDE, lw=2.5)
plt.axhline(1, color="gray", ls="--"); plt.axvline(30, color="gray", ls="--")
plt.xlabel("Ángulo de los cables con la horizontal (°)"); plt.ylabel("T / W")
plt.title("Entre más acostado el cable, más tensión"); plt.ylim(0, 3); plt.show()

## 06 · Momento y sistemas equivalentes

**Momento de una fuerza** respecto a un punto O: su tendencia a hacer girar. En el plano, con (x, y) el punto donde
actúa la fuerza medido desde O:

$$M_O = F\cdot d = x\,F_y - y\,F_x \qquad(\text{antihorario positivo})$$

La segunda forma es el **teorema de Varignon**: el momento de una fuerza es la suma de los momentos de sus componentes.

**Carga distribuida trapezoidal** de $w_1$ a $w_2$ en una longitud L: se reemplaza por una fuerza igual a su área,
aplicada en el centroide del área:

$$R = \frac{(w_1+w_2)L}{2} \qquad \bar x = \frac{L(w_1+2w_2)}{3(w_1+w_2)}$$

In [ ]:
def momento(F, x, theta):
    # Fuerza F aplicada a una distancia x del pivote sobre una barra horizontal,
    # formando un ángulo θ con la barra (medido en sentido antihorario).
    Fx, Fy = F * np.cos(rad(theta)), F * np.sin(rad(theta))
    d = x * abs(np.sin(rad(theta)))      # brazo de palanca: distancia perpendicular
    M = x * Fy - 0 * Fx                  # Varignon con y = 0 (el punto está sobre la barra)
    sentido = "antihorario" if M > 0 else "horario" if M < 0 else "no gira"
    print(f"F = {F} kN, x = {x} m, θ = {theta}° → d = {d:.3f} m, M_O = {M:.3f} kN·m ({sentido})")
    return M

momento(F=4, x=1.5, theta=60)
momento(F=4, x=1.5, theta=90)     # perpendicular: brazo máximo
momento(F=4, x=1.5, theta=0)      # la fuerza pasa por O: no hace momento

def carga_trapezoidal(w1, w2, L):
    # Se descompone en un rectángulo y un triángulo, se suman sus fuerzas y se ubica el centroide.
    w_min, dw = min(w1, w2), abs(w2 - w1)
    R_rect, x_rect = w_min * L, L / 2
    R_tri = dw * L / 2
    x_tri = 2 * L / 3 if w2 >= w1 else L / 3          # el centroide del triángulo está a L/3 del lado alto
    R = R_rect + R_tri
    x_bar = (R_rect * x_rect + R_tri * x_tri) / R
    x_formula = L * (w1 + 2 * w2) / (3 * (w1 + w2))
    print(f"\nCarga de {w1} a {w2} kN/m en {L} m:")
    print(f"  rectángulo {R_rect:.2f} kN en x = {x_rect:.3f} m · triángulo {R_tri:.2f} kN en x = {x_tri:.3f} m")
    print(f"  R = {R:.3f} kN en x̄ = {x_bar:.3f} m (fórmula del trapecio: {x_formula:.3f} m)")
    return R, x_bar

carga_trapezoidal(4, 10, 6);

## 07 · Equilibrio del cuerpo rígido: reacciones de una viga

Una viga, además de no trasladarse, **no debe girar**. Son tres ecuaciones: $\sum F_x = 0$, $\sum F_y = 0$, $\sum M = 0$.

| Apoyo | Reacciones |
|---|---|
| Rodillo | 1 (perpendicular a la superficie) |
| Pasador | 2 (Ax, Ay) |
| Empotramiento | 3 (Ax, Ay, M) |

Con 3 reacciones la viga es **isostática**. El algoritmo: momentos en A para hallar $R_B$, luego $\sum F_y$ para $R_A$.

**Qué hace el código:** la función `resolver_viga` es el **mismo motor** que usa la página web. Recibe la longitud,
el tipo de apoyo y una lista de cargas (puntuales o distribuidas en un tramo), y devuelve las reacciones y dos funciones:
`V(x)` (cortante) y `M(x)` (momento), que se usan en la unidad 09.

In [ ]:
def resolver_viga(L, cargas, tipo="simple", xB=None):
    # tipo: "simple" (pasador en 0 y rodillo en xB) o "empotrada" (empotramiento en x = 0)
    # cargas: lista de diccionarios {"P": kN, "x": m} o {"w": kN/m, "x1": m, "x2": m}
    xB = L if xB is None else xB
    # 1) Resultante de cada carga y su momento respecto a A (x = 0)
    P_total, M_A = 0.0, 0.0
    for c in cargas:
        if "P" in c:
            P_total += c["P"]; M_A += c["P"] * c["x"]
        else:
            W = c["w"] * (c["x2"] - c["x1"])           # resultante de la distribuida
            xc = (c["x1"] + c["x2"]) / 2               # actúa en el centro del tramo
            P_total += W; M_A += W * xc
    # 2) Reacciones por equilibrio
    if tipo == "empotrada":
        RA, RB, MA = P_total, 0.0, M_A                 # ΣFy y ΣM_A en el empotramiento
        apoyos = [(0.0, RA)]
    else:
        RB = M_A / xB                                  # ΣM_A = 0 → R_B
        RA = P_total - RB                              # ΣFy = 0 → R_A
        MA = 0.0
        apoyos = [(0.0, RA), (xB, RB)]
    # 3) Funciones de fuerza interna (método del corte, tramo izquierdo)
    def V(x):
        v = sum(R for xr, R in apoyos if xr <= x)
        for c in cargas:
            if "P" in c and c["x"] <= x: v -= c["P"]
            if "w" in c and x > c["x1"]: v -= c["w"] * (min(x, c["x2"]) - c["x1"])
        return v
    def M(x):
        m = -MA + sum(R * (x - xr) for xr, R in apoyos if xr <= x)
        for c in cargas:
            if "P" in c and c["x"] <= x: m -= c["P"] * (x - c["x"])
            if "w" in c and x > c["x1"]:
                if x <= c["x2"]: m -= c["w"] * (x - c["x1"]) ** 2 / 2
                else: m -= c["w"] * (c["x2"] - c["x1"]) * (x - (c["x1"] + c["x2"]) / 2)
        return m
    return {"RA": RA, "RB": RB, "MA": MA, "V": V, "M": M, "L": L, "P_total": P_total, "xB": xB, "tipo": tipo}

# Ejemplo: viga de 6 m, P = 20 kN a 2 m de A y w = 8 kN/m en toda la luz
viga = resolver_viga(6, [{"P": 20, "x": 2}, {"w": 8, "x1": 0, "x2": 6}])
print(f"R_A = {viga['RA']:.3f} kN   R_B = {viga['RB']:.3f} kN")
print(f"Comprobación: R_A + R_B = {viga['RA'] + viga['RB']:.3f} kN = ΣP = {viga['P_total']:.3f} kN")
print(f"Momento en el apoyo B (debe ser 0): {viga['M'](6):.2e} kN·m")

**Más casos:** viga con voladizo (el apoyo B no está en el extremo) y viga empotrada (voladizo de balcón).

In [ ]:
v2 = resolver_viga(8, [{"P": 15, "x": 3}, {"P": 10, "x": 8}], xB=6.5)      # voladizo de 1,5 m
print(f"Con voladizo:  R_A = {v2['RA']:.3f} kN   R_B = {v2['RB']:.3f} kN")
v3 = resolver_viga(2, [{"P": 5, "x": 2}], tipo="empotrada")                 # balcón de 2 m
print(f"Empotrada:     A_y = {v3['RA']:.3f} kN   M_A = {v3['MA']:.3f} kN·m (antihorario)")

# Parte III · Análisis estructural

## 08 · Armaduras: método de nudos en forma matricial

Una armadura son barras unidas en nudos. Si las cargas llegan solo a los nudos, cada barra solo se estira (**tensión, +**)
o se acorta (**compresión, −**).

**Método de nudos:** cada nudo da dos ecuaciones ($\sum F_x = 0$, $\sum F_y = 0$). Con j nudos hay 2j ecuaciones para
b fuerzas de barra y r = 3 reacciones. La armadura es isostática si $b + r = 2j$.

**Qué hace el código:**
1. `geometria_cercha` arma los nudos y las barras de una cercha Warren, Pratt o Howe.
2. `resolver_cercha` escribe las 2j ecuaciones en una matriz y la resuelve con `np.linalg.solve`. En cada nudo, una
   barra que sale hacia otro nudo aporta su vector unitario: así se supone tensión y el signo del resultado corrige.
3. `dibujar_cercha` pinta cada barra en rojo (tensión) o azul (compresión), con grosor proporcional a la fuerza.

In [ ]:
def geometria_cercha(tipo, L, n, h):
    # Nudos inferiores: 0..n (A, B, C...). Nudos superiores: a continuación.
    nudos = [(i * L / n, 0.0) for i in range(n + 1)]
    barras = [(i, i + 1) for i in range(n)]                                # cordón inferior
    if tipo == "warren":
        nudos += [((i + 0.5) * L / n, h) for i in range(n)]               # nudos de arriba a mitad de panel
        barras += [(n + 1 + i, n + 2 + i) for i in range(n - 1)]          # cordón superior
        for i in range(n):
            barras += [(i, n + 1 + i), (n + 1 + i, i + 1)]                # diagonales en zigzag
    else:
        nudos += [(i * L / n, h) for i in range(n + 1)]                   # nudos de arriba sobre cada nudo de abajo
        barras += [(n + 1 + i, n + 2 + i) for i in range(n)]              # cordón superior
        barras += [(i, n + 1 + i) for i in range(n + 1)]                  # montantes verticales
        for i in range(n):
            izq = i < n / 2
            if tipo == "pratt":   # diagonales bajan hacia el centro (trabajan a tensión)
                barras.append((n + 1 + i, i + 1) if izq else (n + 2 + i, i))
            else:                 # howe: diagonales suben hacia el centro (trabajan a compresión)
                barras.append((i, n + 2 + i) if izq else (i + 1, n + 1 + i))
    return {"nudos": np.array(nudos), "barras": barras, "n": n, "L": L, "h": h, "tipo": tipo}

def cargas_tablero(g, w, P_extra=0, nudo_extra=1):
    # La carga repartida w (kN/m) se lleva a los nudos inferiores por área tributaria:
    # cada nudo interior recibe w·L/n y cada extremo la mitad.
    cargas = np.zeros((len(g["nudos"]), 2))
    for i in range(g["n"] + 1):
        trib = g["L"] / (2 * g["n"]) if i in (0, g["n"]) else g["L"] / g["n"]
        cargas[i, 1] -= w * trib
    cargas[nudo_extra, 1] -= P_extra
    return cargas

def resolver_cercha(g, cargas):
    N, B = g["nudos"], g["barras"]
    nb, nn = len(B), len(N)
    assert nb + 3 == 2 * nn, "La cercha no es isostática: b + r ≠ 2j"
    A = np.zeros((2 * nn, nb + 3))           # incógnitas: fuerzas de barra + Ax, Ay, By
    for k, (i, j) in enumerate(B):
        u = (N[j] - N[i]) / np.linalg.norm(N[j] - N[i])     # vector unitario de i hacia j
        A[2 * i:2 * i + 2, k] += u            # en el nudo i la barra jala hacia j (tensión supuesta)
        A[2 * j:2 * j + 2, k] -= u            # en el nudo j jala hacia i
    pasador, rodillo = 0, g["n"]
    A[2 * pasador, nb] = 1                    # Ax en el pasador
    A[2 * pasador + 1, nb + 1] = 1            # Ay en el pasador
    A[2 * rodillo + 1, nb + 2] = 1            # By en el rodillo
    b = -cargas.reshape(-1)                   # las cargas pasan al otro lado de la ecuación
    x = np.linalg.solve(A, b)
    return x[:nb], x[nb], x[nb + 1], x[nb + 2]   # fuerzas, Ax, Ay, By

def nombre(i):
    return "ABCDEFGHIJKLMNOPQRSTUVWXYZ"[i]

def dibujar_cercha(g, F, cargas):
    N, B = g["nudos"], g["barras"]
    Fmax = np.abs(F).max()
    fig, ax = plt.subplots(figsize=(10, 3.6))
    for (i, j), f in zip(B, F):
        cero = abs(f) < 1e-6
        color = "gray" if cero else (ROJO_TENSION if f > 0 else AZUL_COMPRESION)
        ax.plot(*zip(N[i], N[j]), color=color, lw=1.2 if cero else 1.5 + 6 * abs(f) / Fmax, ls="--" if cero else "-")
        xm, ym = (N[i] + N[j]) / 2
        ax.text(xm, ym, f"{f:.1f}", ha="center", va="center", fontsize=8, color=color,
                bbox=dict(fc="white", ec="none", alpha=0.8, pad=0.5))
    for k, (x, y) in enumerate(N):
        ax.plot(x, y, "o", color="k", ms=4); ax.text(x, y + 0.12 * g["h"], nombre(k), ha="center", fontsize=8)
    for k, (cx, cy) in enumerate(cargas):
        if cy < 0:
            ax.annotate("", xy=N[k], xytext=(N[k][0], N[k][1] + 0.35 * g["h"]),
                        arrowprops=dict(arrowstyle="-|>", color=NARANJA, lw=1.5))
    ax.set_aspect("equal"); ax.set_title(f"Cercha {g['tipo'].capitalize()}: rojo = tensión, azul = compresión (kN)")
    ax.axis("off"); plt.show()

# Ejemplo: Pratt de 12 m, 6 paneles, 3 m de altura, 6 kN/m de tablero
g = geometria_cercha("pratt", L=12, n=6, h=3)
cargas = cargas_tablero(g, w=6)
F, Ax, Ay, By = resolver_cercha(g, cargas)
print(f"b + r = {len(g['barras'])} + 3 = {len(g['barras']) + 3}   2j = {2 * len(g['nudos'])}  ✓ isostática")
print(f"Reacciones: Ay = {Ay:.2f} kN, By = {By:.2f} kN, Ax = {Ax:.2f} kN")
dibujar_cercha(g, F, cargas)
for (i, j), f in zip(g["barras"], F):
    estado = "fuerza cero" if abs(f) < 1e-6 else ("tensión" if f > 0 else "compresión")
    print(f"  {nombre(i)}{nombre(j)}: {f:8.2f} kN  {estado}")

**Comprobación por el método de secciones.** Al cortar la cercha, la fuerza en un cordón es el momento de la
"viga equivalente" en el nudo opuesto dividido por la altura: $F_{cordón} = M_{nudo}/h$. El cordón más cargado debe
coincidir con el momento máximo entre h.

In [ ]:
def comprobar_secciones(g, F, cargas, Ay):
    N, n, h = g["nudos"], g["n"], g["h"]
    # Momento de la viga equivalente en cada nudo inferior (reacción por la izquierda menos cargas a la izquierda)
    M_nudos = []
    for j in range(n + 1):
        xj = N[j, 0]
        m = Ay * xj + sum(cargas[i, 1] * (xj - N[i, 0]) for i in range(n + 1) if N[i, 0] < xj)
        M_nudos.append(m)
    cordones = [k for k, (i, j) in enumerate(g["barras"]) if N[i, 1] == N[j, 1]]   # barras horizontales
    max_cordon = max(abs(F[k]) for k in cordones)
    print(f"M máx / h = {max(M_nudos):.3f} / {h} = {max(M_nudos) / h:.3f} kN")
    print(f"Cordón más cargado (método de nudos) = {max_cordon:.3f} kN → los dos métodos coinciden")

comprobar_secciones(g, F, cargas, Ay)

**Pruébalo:** cambia el tipo de cercha, la altura y la carga. Mira cómo bajan las fuerzas de los cordones cuando sube la altura.

In [ ]:
@interact(tipo=Dropdown(options=["warren", "pratt", "howe"], value="warren", description="Tipo"),
          n=Dropdown(options=[4, 6, 8], value=4, description="Paneles"),
          L=IntSlider(value=12, min=6, max=30, description="Luz (m)"),
          h=FloatSlider(value=3, min=1, max=6, step=0.25, description="Altura (m)"),
          w=FloatSlider(value=6, min=0, max=20, step=0.5, description="w (kN/m)"))
def _(tipo, n, L, h, w):
    g = geometria_cercha(tipo, L, n, h)
    c = cargas_tablero(g, w)
    F, Ax, Ay, By = resolver_cercha(g, c)
    dibujar_cercha(g, F, c)
    print(f"Tensión máx. = {F.max():.2f} kN   Compresión máx. = {F.min():.2f} kN   Ay = By = {Ay:.2f} kN")

## 09 · Fuerzas internas en vigas: diagramas de cortante y momento

Un corte imaginario en x muestra tres fuerzas internas: **N** (normal), **V** (cortante) y **M** (momento flector).
Convención: M positivo dobla la viga cóncava hacia arriba (fibras de abajo en tensión).

$$\frac{dV}{dx} = -w \qquad \frac{dM}{dx} = V$$

Por eso el **momento máximo** está donde V cruza por cero.

**Qué hace el código:** toma la viga de la unidad 07, evalúa V(x) y M(x) en muchos puntos (incluidos los puntos exactos
de cada carga, para no perder los saltos) y dibuja los dos diagramas marcando el máximo.

In [ ]:
def diagramas(viga, x_corte=None):
    L = viga["L"]
    xs = np.sort(np.r_[np.linspace(0, L, 801), [0, L]])     # puntos donde se evalúan V y M
    Vs = np.array([viga["V"](x) for x in xs])
    Ms = np.array([viga["M"](x) for x in xs])
    i = Ms.argmax()
    fig, (a1, a2) = plt.subplots(2, 1, figsize=(8, 5.5), sharex=True)
    a1.fill_between(xs, Vs, color=MORADO, alpha=0.2); a1.plot(xs, Vs, color=MORADO, lw=2)
    a1.set_ylabel("V (kN)"); a1.set_title("Fuerza cortante V(x)")
    a2.fill_between(xs, Ms, color=VERDE, alpha=0.2); a2.plot(xs, Ms, color=VERDE, lw=2)
    a2.plot(xs[i], Ms[i], "o", color=VERDE); a2.annotate(f" Mmáx = {Ms[i]:.2f} kN·m\n en x = {xs[i]:.2f} m", (xs[i], Ms[i]))
    a2.set_ylabel("M (kN·m)"); a2.set_xlabel("x (m)"); a2.set_title("Momento flector M(x)")
    if x_corte is not None:
        for a in (a1, a2): a.axvline(x_corte, color=NARANJA, ls="--")
        print(f"En el corte x = {x_corte} m: V = {viga['V'](x_corte):.3f} kN, M = {viga['M'](x_corte):.3f} kN·m")
    plt.tight_layout(); plt.show()
    print(f"Mmáx = {Ms[i]:.3f} kN·m en x = {xs[i]:.3f} m · |V|máx = {np.abs(Vs).max():.3f} kN")

diagramas(viga, x_corte=1.5)        # la viga de la unidad 07

In [ ]:
@interact(L=FloatSlider(value=6, min=2, max=12, step=0.5, description="L (m)"),
          P=FloatSlider(value=20, min=0, max=50, step=1, description="P (kN)"),
          a=FloatSlider(value=0.33, min=0, max=1, step=0.01, description="x de P / L"),
          w=FloatSlider(value=8, min=0, max=20, step=0.5, description="w (kN/m)"))
def _(L, P, a, w):
    cargas = []
    if P > 0: cargas.append({"P": P, "x": a * L})
    if w > 0: cargas.append({"w": w, "x1": 0, "x2": L})
    v = resolver_viga(L, cargas)
    print(f"R_A = {v['RA']:.2f} kN   R_B = {v['RB']:.2f} kN")
    diagramas(v)

## 10 · Fricción seca

La fricción se opone al deslizamiento y llega como máximo a $F_{máx} = \mu_s N$ (ley de Coulomb).
Al jalar un bloque con un cable a θ sobre la horizontal, la componente vertical reduce la normal:

$$N = W - P\sin\theta \qquad P\cos\theta = \mu_s N \;\Rightarrow\; P = \frac{\mu_s W}{\cos\theta + \mu_s\sin\theta}$$

Derivando e igualando a cero, el **ángulo óptimo** es $\tan\theta = \mu_s$, con $P_{mín} = \mu_s W/\sqrt{1+\mu_s^2}$.
Por eso no conviene jalar horizontal: un poco de inclinación alivia la fricción.

In [ ]:
def friccion(W, mu, theta):
    t = rad(theta)
    P = mu * W / (np.cos(t) + mu * np.sin(t))       # fuerza para iniciar el movimiento
    th_opt = np.degrees(np.arctan(mu))              # ángulo óptimo
    P_min = mu * W / np.sqrt(1 + mu ** 2)
    print(f"W = {W} kN, μs = {mu}, θ = {theta}° → P = {P:.3f} kN")
    print(f"Ángulo óptimo θ = atan(μs) = {th_opt:.1f}° → P mínima = {P_min:.3f} kN "
          f"({(1 - P_min / (mu * W)) * 100:.1f} % menos que jalando horizontal)")
    ang = np.linspace(0, 60, 300)
    Ps = mu * W / (np.cos(rad(ang)) + mu * np.sin(rad(ang)))
    plt.figure(figsize=(6, 3.5)); plt.plot(ang, Ps, color=NARANJA, lw=2.5)
    plt.axvline(th_opt, color=VERDE, ls="--", label=f"óptimo {th_opt:.1f}°")
    plt.plot(theta, P, "o", color="k", label=f"θ = {theta}°")
    plt.xlabel("Ángulo del cable θ (°)"); plt.ylabel("P para iniciar (kN)"); plt.legend(); plt.show()

# μs típicos y aproximados: concreto/concreto ≈ 0,6 · concreto/suelo ≈ 0,45 · madera/concreto ≈ 0,5 · acero/acero ≈ 0,3
friccion(W=10, mu=0.45, theta=10)

# Parte IV · Geometría y síntesis

## 11 · Centroide y momento de inercia de secciones compuestas

Una viga T o un perfil I se dividen en rectángulos. Con y medido desde la base:

$$\bar y = \frac{\sum A_i\,y_i}{\sum A_i} \qquad I = \sum\left(\bar I_i + A_i d_i^2\right), \quad \bar I_i = \frac{b_i h_i^3}{12}, \quad d_i = y_i - \bar y$$

El término $A_i d_i^2$ es el **teorema de ejes paralelos** (Steiner): traslada la inercia de cada parte al eje común.

**Qué hace el código:** arma la tabla de partes (área, centroide, inercia propia, distancia y término de Steiner),
calcula ȳ e I, y dibuja la sección con su eje centroidal.

In [ ]:
def seccion(forma, bf=80, tf=10, bw=25, h=50):
    # Todas las medidas en cm. Cada parte: (nombre, ancho b, alto, y del centroide desde la base)
    if forma == "T":
        partes = [("Ala", bf, tf, h - tf / 2), ("Alma", bw, h - tf, (h - tf) / 2)]
    elif forma == "I":
        partes = [("Ala sup.", bf, tf, h - tf / 2), ("Alma", bw, h - 2 * tf, h / 2), ("Ala inf.", bf, tf, tf / 2)]
    else:
        partes = [("Rectángulo", bw, h, h / 2)]
    A = np.array([b * hh for _, b, hh, _ in partes])
    y = np.array([yy for *_, yy in partes])
    yb = (A * y).sum() / A.sum()                                   # centroide
    I0 = np.array([b * hh ** 3 / 12 for _, b, hh, _ in partes])    # inercia propia de cada rectángulo
    d = y - yb
    I = (I0 + A * d ** 2).sum()                                    # Steiner
    print(f"{'Parte':10s} {'A (cm²)':>9s} {'y (cm)':>8s} {'A·y':>10s} {'Ī (cm⁴)':>11s} {'d (cm)':>8s} {'A·d²':>11s}")
    for (nm, b, hh, yy), a_, i0, dd in zip(partes, A, I0, d):
        print(f"{nm:10s} {a_:9.1f} {yy:8.2f} {a_ * yy:10.1f} {i0:11.0f} {dd:8.2f} {a_ * dd ** 2:11.0f}")
    print(f"\nÁrea = {A.sum():.1f} cm²   ȳ = {yb:.2f} cm   I = {I:,.0f} cm⁴ = {I / 1e8:.6f} m⁴")
    print(f"El término A·d² aporta el {(A * d ** 2).sum() / I * 100:.0f} % de la inercia")
    # Dibujo de la sección
    fig, ax = plt.subplots(figsize=(4, 4))
    for k, (nm, b, hh, yy) in enumerate(partes):
        ax.add_patch(plt.Rectangle((-b / 2, yy - hh / 2), b, hh, fc=[AZUL, MORADO][k % 2], alpha=0.35, ec="k"))
    ax.axhline(yb, color=NARANJA, ls="--", lw=2); ax.text(max(bf, bw) / 2, yb, f"  ȳ = {yb:.1f} cm", color=NARANJA, va="bottom")
    ax.set_xlim(-max(bf, bw) / 2 - 10, max(bf, bw) / 2 + 30); ax.set_ylim(-5, h + 5); ax.set_aspect("equal")
    ax.set_title(f"Sección {forma}"); plt.show()
    return yb, I

seccion("T", bf=80, tf=10, bw=25, h=50);

## 12 · Caso integrador: izaje de una viga

Une las unidades anteriores: **peso propio** (unidad 02: $W = \gamma V$, concreto reforzado ≈ 24 kN/m³),
**componentes** (unidad 03) y **equilibrio del gancho** (unidad 05):

$$\sum F_y:\; 2T\sin\theta - W = 0 \;\Rightarrow\; T = \frac{W}{2\sin\theta}$$

Al final se compara la tensión con la **capacidad de trabajo (WLL)** de cada eslinga. La componente horizontal
$H = T\cos\theta$ comprime la viga de punta a punta.

In [ ]:
def izaje(L, b_cm, h_cm, theta, WLL, gamma=24):
    V = L * (b_cm / 100) * (h_cm / 100)                 # volumen (m³)
    W = gamma * V                                       # peso (kN)
    T = W / (2 * np.sin(rad(theta)))                    # tensión en cada eslinga
    H = T * np.cos(rad(theta))                          # compresión axial en la viga
    uso = T / WLL
    if theta >= 60: angulo = "recomendado"
    elif theta >= 45: angulo = "aceptable, con cuidado"
    elif theta >= 30: angulo = "precaución: tensión alta"
    else: angulo = "evitar: tensión excesiva"
    print(f"Viga {L} m · {b_cm}×{h_cm} cm → V = {V:.3f} m³, W = {W:.2f} kN (≈ {W / 9.81:.2f} t)")
    print(f"θ = {theta}° ({angulo}) → T = {T:.2f} kN por eslinga ({T / W:.2f} veces el peso)")
    print(f"Compresión en la viga H = {H:.2f} kN")
    print(f"T / WLL = {T:.2f} / {WLL} = {uso:.2f} → {'CUMPLE' if uso <= 1 else 'NO CUMPLE: usar eslingas de mayor capacidad o abrir menos'}")

izaje(L=6, b_cm=30, h_cm=50, theta=60, WLL=20)
print()
izaje(L=6, b_cm=30, h_cm=50, theta=25, WLL=20)

## Banco de ejercicios

Esta celda genera problemas con datos aleatorios. Ejecútala, resuelve en el cuaderno y compara con la respuesta,
que aparece al ejecutar la celda siguiente.

In [ ]:
import random

def ejercicio():
    tipo = random.choice(["nudo", "viga", "momento", "friccion"])
    if tipo == "nudo":
        W, a = random.choice([6, 8, 10, 12]), random.choice([30, 40, 45, 50, 60])
        return (f"Un andamio de {W} kN cuelga de dos cables simétricos a {a}° de la horizontal. ¿Tensión de cada cable?",
                W / (2 * np.sin(rad(a))), "kN")
    if tipo == "viga":
        L, P, a = random.choice([4, 5, 6, 8]), random.choice([10, 12, 15, 20]), random.choice([1, 1.5, 2])
        v = resolver_viga(L, [{"P": P, "x": a}])
        return (f"Viga simplemente apoyada de {L} m con P = {P} kN a {a} m de A. ¿Cuánto vale R_A?", v["RA"], "kN")
    if tipo == "momento":
        F, x, t = random.choice([2, 3, 4, 5]), random.choice([0.8, 1.2, 1.5, 2]), random.choice([30, 45, 60, 75])
        return (f"Una fuerza de {F} kN actúa a {x} m del pivote formando {t}° con la barra. ¿Momento?",
                F * x * np.sin(rad(t)), "kN·m")
    W, mu, t = random.choice([5, 8, 10, 15]), random.choice([0.3, 0.4, 0.5, 0.6]), random.choice([0, 15, 20, 30])
    return (f"Un bloque de {W} kN se jala a {t}° con μs = {mu}. ¿Fuerza para iniciar el movimiento?",
            mu * W / (np.cos(rad(t)) + mu * np.sin(rad(t))), "kN")

pregunta, respuesta, unidad = ejercicio()
print(pregunta)

In [ ]:
print(f"Respuesta: {respuesta:.3f} {unidad}")

## Verificación del motor de cálculo

Cada línea compara el resultado del código con la fórmula analítica de los libros (Hibbeler, Beer).
Si alguna falla, `assert` detiene la celda y muestra cuál. Esto es lo que da confianza en los resultados de
las unidades anteriores.

In [ ]:
def verificar(nombre, calculado, analitico, tol=1e-8):
    ok = abs(calculado - analitico) < tol
    print(f"{'OK ' if ok else 'FALLA'} {nombre}: {calculado:.6f} vs {analitico:.6f}")
    assert ok, nombre

def M_max(v):
    xs = np.sort(np.r_[np.linspace(0, v["L"], 801), [0, v["L"]]])
    return max(v["M"](x) for x in xs)

verificar("Viga 6 m, P = 12 kN al centro · Mmáx = PL/4", M_max(resolver_viga(6, [{"P": 12, "x": 3}])), 18)
verificar("Viga 6 m, w = 8 kN/m · R_A = wL/2", resolver_viga(6, [{"w": 8, "x1": 0, "x2": 6}])["RA"], 24)
verificar("Viga 6 m, w = 8 kN/m · Mmáx = wL²/8", M_max(resolver_viga(6, [{"w": 8, "x1": 0, "x2": 6}])), 36)
verificar("Viga 6 m, P = 15 kN a 2 m · Mmáx = Pab/L", resolver_viga(6, [{"P": 15, "x": 2}])["M"](2), 20)
verificar("Voladizo 2 m, P = 5 kN · M_A = PL", resolver_viga(2, [{"P": 5, "x": 2}], tipo="empotrada")["MA"], 10)
verificar("Voladizo 3 m, w = 2 kN/m · M_A = wL²/2", resolver_viga(3, [{"w": 2, "x1": 0, "x2": 3}], tipo="empotrada")["MA"], 9)

g = geometria_cercha("warren", 12, 3, 3)
c = np.zeros((len(g["nudos"]), 2)); c[1, 1] = -20; c[2, 1] = -10
F, Ax, Ay, By = resolver_cercha(g, c)
idx = lambda a, b: next(k for k, br in enumerate(g["barras"]) if set(br) == {a, b})
Dy = (20 * 4 + 10 * 8) / 12; Ay_ = 30 - Dy; sn = 3 / np.sqrt(13)
verificar("Warren 12 m · EF por secciones", F[idx(4, 5)], -4 * Ay_ / 3)
verificar("Warren 12 m · BC por secciones", F[idx(1, 2)], (6 * Ay_ - 2 * 20) / 3)
verificar("Warren 12 m · BF por secciones", F[idx(1, 5)], (20 - Ay_) / sn)
g2 = geometria_cercha("howe", 12, 6, 3); F2, *_ = resolver_cercha(g2, cargas_tablero(g2, 10))
k_central = g2["barras"].index((2, 3))          # cordón inferior del panel central (nudos C y D)
verificar("Howe 12 m, w = 10 · cordón inferior central = (wL²/8)/h", F2[k_central], 10 * 12 ** 2 / 8 / 3)
verificar("Fricción · P mínima en θ = atan(μ)", 0.45 * 10 / (np.cos(np.arctan(0.45)) + 0.45 * np.sin(np.arctan(0.45))), 0.45 * 10 / np.sqrt(1 + 0.45 ** 2))
print("\nTodas las verificaciones pasaron.")

---
**Referencias:** Hibbeler, R. C. (2016). *Ingeniería mecánica: Estática* (14.ª ed.). Pearson ·
Beer, F. P., Johnston, E. R., Mazurek, D. F. y Cornwell, P. J. (2013). *Mecánica vectorial para ingenieros: Estática* (10.ª ed.). McGraw-Hill ·
NSR-10 (2010).

Herramienta de estudio: los valores son supuestos académicos y no reemplazan el diseño ni la firma de un ingeniero.